# 01 - Exploratory Data Analysis: UCI Hydraulic Systems Dataset

**Goal**: 데이터 로드 → 라벨 분포 확인 → 신호 시각화 → 특성 추출 → LDA 5-fold CV baseline

**Dataset**: UCI Condition Monitoring of Hydraulic Systems (2205 cycles, 17 sensors, 4 fault targets)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_all_sensors, load_labels, get_data_dir, SENSOR_SPECS, LABEL_DESCRIPTIONS
from src.features import extract_all_features, time_domain_features, TIME_FEATURE_NAMES
from src.visualization import (
    plot_label_distribution, plot_cycle_examples, plot_class_mean_std,
    plot_psd_by_class, plot_multifault_heatmap, plot_correlation_matrix, plot_confusion_matrix
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
print('Imports OK')

In [ ]:
data_dir = get_data_dir()
print(f'Data directory: {data_dir}')
print(f'Files: {sorted([f.name for f in data_dir.iterdir() if f.suffix == ".txt"])}')

## 1. Load Data

In [ ]:
sensors = load_all_sensors(data_dir)

In [ ]:
labels = load_labels(data_dir)
print(f'Labels shape: {labels.shape}')
labels.head()

In [ ]:
print('=== Sensor Summary ===')
for name, spec in SENSOR_SPECS.items():
    arr = sensors[name]
    print(f'{name:5s} | {spec["physical"]:30s} | {spec["sampling_hz"]:4d} Hz | shape {arr.shape} | range [{arr.min():.2f}, {arr.max():.2f}] {spec["unit"]}')

print(f'\n=== Label Value Counts ===')
for col in labels.columns:
    desc = LABEL_DESCRIPTIONS[col]
    print(f'\n{col}:')
    for val, cnt in labels[col].value_counts().sort_index().items():
        label_name = desc['values'].get(val, '?')
        print(f'  {val:4d} ({label_name:30s}): {cnt:5d} ({cnt/len(labels)*100:.1f}%)')

## 2. Label Distribution

In [ ]:
fig = plot_label_distribution(labels)
plt.show()

## 3. Multi-Fault Co-occurrence

유압 시스템에서는 하나의 결함이 다른 컴포넌트에도 영향을 미칠 수 있다.  
아래 heatmap은 서로 다른 타깃 라벨 간의 **동시 발생 빈도**를 보여준다.

In [ ]:
fig = plot_multifault_heatmap(labels)
plt.show()

**관찰 포인트**:
- 특정 결함 조합이 자주/드물게 나타나는 패턴이 있는지 확인
- 이는 multi-task learning (H4) 에서 label correlation을 활용할 근거가 됨

## 4. Signal Visualization

In [ ]:
fig = plot_cycle_examples(sensors['PS1'], 'PS1', labels['cooler'], n_per_class=3)
plt.show()

In [ ]:
fig = plot_cycle_examples(sensors['VS1'], 'VS1', labels['pump'], n_per_class=3)
plt.show()

In [ ]:
fig = plot_class_mean_std(sensors['PS1'], 'PS1', labels['valve'])
plt.show()

In [ ]:
fig = plot_class_mean_std(sensors['TS1'], 'TS1', labels['cooler'])
plt.show()

## 5. Frequency Domain

In [ ]:
fig = plot_psd_by_class(sensors['PS1'], 'PS1', labels['valve'], fs=100)
plt.show()

In [ ]:
fig = plot_psd_by_class(sensors['VS1'], 'VS1', labels['pump'], fs=1)
plt.show()

## 6. Feature Extraction

In [ ]:
print('Extracting time-domain features from all 17 sensors...')
feature_df = extract_all_features(sensors, labels)
print(f'Feature matrix shape: {feature_df.shape}')
print(f'  Sensor features: {feature_df.shape[1] - len(labels.columns)}')
print(f'  Label columns: {list(labels.columns)}')

In [ ]:
feature_df.head()

In [ ]:
fig = plot_correlation_matrix(feature_df, max_features=40)
plt.show()

## 7. LDA Baseline (Helwig 2015 Replication)

Helwig et al. (2015) 의 원전(原典) 방법론을 재현한다.  
Linear Discriminant Analysis (LDA) 로 각 타깃을 독립적으로 분류하고 5-fold Stratified CV 로 평가한다.

참고: 원 논문의 정확한 전처리/피처를 사용하지 않았으므로 수치가 다를 수 있다.  
Week 2에서 정밀 재현을 진행한다.

### Setup

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

label_cols = ['cooler', 'valve', 'pump', 'accumulator']
feature_cols = [c for c in feature_df.columns if c not in labels.columns]

X = feature_df[feature_cols].values
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lda_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis()),
])

print(f'Feature matrix: {X.shape}')
print(f'Targets: {label_cols}')

In [ ]:
# Cooler
y = labels['cooler'].values
scores = cross_val_score(lda_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'=== Cooler ===')
print(f'LDA 5-fold CV Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)')
print(f'Per-fold: {[f"{s*100:.1f}%" for s in scores]}')

In [ ]:
# Valve
y = labels['valve'].values
scores = cross_val_score(lda_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'=== Valve ===')
print(f'LDA 5-fold CV Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)')
print(f'Per-fold: {[f"{s*100:.1f}%" for s in scores]}')

In [ ]:
# Pump
y = labels['pump'].values
scores = cross_val_score(lda_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'=== Pump ===')
print(f'LDA 5-fold CV Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)')
print(f'Per-fold: {[f"{s*100:.1f}%" for s in scores]}')

In [ ]:
# Accumulator
y = labels['accumulator'].values
scores_accu = cross_val_score(lda_pipeline, X, y, cv=cv, scoring='accuracy')
print(f'=== Accumulator ===')
print(f'LDA 5-fold CV Accuracy: {scores_accu.mean()*100:.2f}% (+/- {scores_accu.std()*100:.2f}%)')
print(f'Per-fold: {[f"{s*100:.1f}%" for s in scores_accu]}')

print('\n' + '='*60)
print('Summary: LDA 5-Fold CV Baseline')
print('='*60)
# Re-run all to collect in one place
results = {}
for target in ['cooler', 'valve', 'pump', 'accumulator']:
    y = labels[target].values
    sc = cross_val_score(lda_pipeline, X, y, cv=cv, scoring='accuracy')
    results[target] = (sc.mean(), sc.std())
    print(f'  {target:15s}: {sc.mean()*100:.2f}% (+/- {sc.std()*100:.2f}%)')
print('='*60)

## Summary & Next Steps

### 이번 EDA에서 확인한 것
1. **데이터 구조**: 2205 cycles, 17 sensors (100/10/1 Hz), 4 fault targets + stable flag
2. **라벨 분포**: 클래스 불균형 존재 (특히 pump의 severe leakage가 적음)
3. **신호 패턴**: 결함 상태에 따라 신호 형태가 시각적으로 구분 가능
4. **복합결함**: 특정 결함 조합이 빈번하게 동시 발생 → multi-task learning 근거
5. **LDA baseline**: 위 표 참고. Week 2에서 이 수치를 넘는 것이 목표

### Week 2 계획
- Helwig 2015 LDA 정밀 재현 (논문 특성 세트 그대로 사용)
- RF, XGBoost, LightGBM, Extra Trees 비교 (`03_baseline_ml.ipynb`)
- 최선의 ML baseline 수치 확보